# Colonel Blotto — Actor-Critic Mixed Strategy Solver

**Game:** 2-player, 3-battlefield Colonel Blotto with equal budgets (B=1) and equal battlefield values.  
**Method:** Actor-Critic with Double Q-Critic and replicator dynamics, extending [Martin & Sandholm (2022)](https://arxiv.org/abs/2211.15936) which used zeroth-order optimization.  
**Analytical equilibrium:** Uniform distribution over a hemisphere inscribed in the 2-simplex.  
Each battlefield's marginal mean = 1/3; symmetric expected payoff = 1.5.

---
**Notebook structure:**
1. Imports & game environment
2. Networks (Actor, Double Q-Critic)
3. Replay buffer
4. Config & convergence criterion
5. Trainer (single shared Actor)
6. Training run & visualisation
7. Final diagnostics
8. Extension: Two independent Actors (symmetry emergence test)

## 1. Imports & Game Environment

In [ ]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from dataclasses import dataclass
from scipy import stats
import matplotlib.pyplot as plt

torch.manual_seed(2024)
np.random.seed(2024)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Colonel Blotto：2玩家，3战场，等预算 B=1，等价值
#
# 动作：a = (a1, a2, a3)，满足 sum=1，a_j >= 0
# 收益：赢得的战场数（出价高者赢，平局各得0.5）
# 均衡期望收益：1.5（对称）
# ══════════════════════════════════════════════════════════════════════

N_BATTLEFIELDS = 3
EQ_PAYOFF      = 1.5


def payoff(a1: torch.Tensor, a2: torch.Tensor):
    """
    a1, a2: (N, 3)
    返回 u1, u2: (N,)
    """
    win1 = (a1 > a2).float() + 0.5 * (a1 == a2).float()
    win2 = (a2 > a1).float() + 0.5 * (a1 == a2).float()
    return win1.sum(dim=1), win2.sum(dim=1)


def simplex_sample(n: int, device=DEVICE) -> torch.Tensor:
    """从标准单纯形均匀采样，用 Dirichlet(1,1,1)"""
    return torch.distributions.Dirichlet(
        torch.ones(N_BATTLEFIELDS, device=device)
    ).sample((n,))


def nash_gap(a1: torch.Tensor, a2: torch.Tensor) -> float:
    """Nash Gap = |E[u1] - 1.5|"""
    with torch.no_grad():
        u1, _ = payoff(a1, a2)
    return abs(u1.mean().item() - EQ_PAYOFF)


# —— 验证 ——
print("=== 收益验证 ===")
_a1 = torch.tensor([[0.5, 0.3, 0.2],   # 赢战场1，输战场2，赢战场3
                     [0.4, 0.4, 0.2],   # 平战场1，平战场2，平战场3
                     [1/3, 1/3, 1/3],   # 完全对称
                     [1.0, 0.0, 0.0]])  # 极端情况
_a2 = torch.tensor([[0.3, 0.4, 0.3],
                     [0.4, 0.4, 0.2],
                     [1/3, 1/3, 1/3],
                     [0.0, 0.5, 0.5]])
_u1, _u2 = payoff(_a1, _a2)
for i in range(4):
    print(f"  a1={[f'{x:.2f}' for x in _a1[i].tolist()]} "
          f"a2={[f'{x:.2f}' for x in _a2[i].tolist()]} "
          f"→ u1={_u1[i]:.2f}  u2={_u2[i]:.2f}")

print("\n=== simplex_sample 验证 ===")
_s = simplex_sample(10000)
print(f"  Shape: {_s.shape}")
print(f"  Sum 应全为1: min={_s.sum(-1).min():.6f}  max={_s.sum(-1).max():.6f}")
print(f"  每维度均值 应接近 1/3: {_s.mean(0).tolist()}")
print(f"  每维度非负: {(_s >= 0).all()}")

## 2. Network Architecture

- **Actor:** noise z ~ N(0, I_d) → softmax → action on simplex  
- **QCritic:** action → Q(a; π_opponent) via MC targets  
- **DoubleQCritic:** two independent critics; conservative estimate = mean − β·|C1−C2|

In [ ]:
# 网络架构
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))


# ── Actor ─────────────────────────────────────────────────────────────
class Actor(nn.Module):
    """
    z ~ N(0, I_d) → a ∈ Simplex（3维，和为1）
    输出用 Softmax 保证和为1且非负
    """
    def __init__(self, noise_dim=8):
        super().__init__()
        self.noise_dim = noise_dim
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 128), Mish(),
            nn.Linear(128, 128),       Mish(),
            nn.Linear(128, 64),        Mish(),
            nn.Linear(64, N_BATTLEFIELDS),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, z):
        """z: (N, noise_dim) → a: (N, 3)"""
        return F.softmax(self.net(z), dim=-1)

    def sample(self, n, device=DEVICE):
        z = torch.randn(n, self.noise_dim, device=device)
        return self.forward(z)


# ── QCritic ───────────────────────────────────────────────────────────
class QCritic(nn.Module):
    """
    输入：a1 ∈ Simplex（3维）
    输出：Q(a1; π) = E_{a2~π}[u1(a1, a2)]
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_BATTLEFIELDS, 128), nn.LayerNorm(128), Mish(),
            nn.Linear(128, 128),            nn.LayerNorm(128), Mish(),
            nn.Linear(128, 64),             Mish(),
            nn.Linear(64, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, a1):
        """a1: (N, 3) → Q: (N,)"""
        return self.net(a1).squeeze(-1)


# ── DoubleQCritic ─────────────────────────────────────────────────────
class DoubleQCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = QCritic()
        self.c2 = QCritic()

    def forward(self, a1):
        return self.c1(a1), self.c2(a1)

    def mean_Q(self, a1):
        return (self.c1(a1) + self.c2(a1)) / 2.

    def conservative_Q(self, a1, beta=0.3):
        q1, q2 = self.c1(a1), self.c2(a1)
        return (q1 + q2) / 2. - beta * (q1 - q2).abs()


# —— 验证 ——
print("=== 网络验证 ===")
_actor = Actor(8).to(DEVICE)   # 加 .to(DEVICE)
_dc    = DoubleQCritic().to(DEVICE)   # 加 .to(DEVICE)
print(f"Actor  参数量: {sum(p.numel() for p in _actor.parameters()):,}")
print(f"Critic 参数量: {sum(p.numel() for p in _dc.parameters()):,}")

_xs = _actor.sample(1024)
print(f"\nActor 输出形状: {_xs.shape}  (期望 [1024, 3])")
print(f"Actor 输出和:   min={_xs.sum(-1).min():.6f}  max={_xs.sum(-1).max():.6f}  (应全为1)")
print(f"Actor 输出非负: {(_xs >= 0).all()}")

_q1, _q2 = _dc(_xs)
print(f"\nCritic 输出形状: {_q1.shape}  (期望 [1024])")
print(f"Q1 范围: [{_q1.min():.3f}, {_q1.max():.3f}]")
print(f"Q2 范围: [{_q2.min():.3f}, {_q2.max():.3f}]")

## 3. Replay Buffer

Ring buffer storing (a1, a2, u1, u2). Sampling mixes recent transitions (50%) with historical (50%) to track the current policy while preventing forgetting.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=200_000, recent_ratio=0.5, device=DEVICE):
        self.capacity     = capacity
        self.recent_ratio = recent_ratio
        self.device       = device
        self._a1  = np.zeros((capacity, N_BATTLEFIELDS), np.float32)
        self._a2  = np.zeros((capacity, N_BATTLEFIELDS), np.float32)
        self._u1  = np.zeros(capacity, np.float32)
        self._u2  = np.zeros(capacity, np.float32)
        self._ptr  = 0
        self._size = 0

    def push(self, a1, a2, u1, u2):
        a1 = a1.detach().cpu().numpy()
        a2 = a2.detach().cpu().numpy()
        u1 = u1.detach().cpu().numpy().flatten()
        u2 = u2.detach().cpu().numpy().flatten()
        n   = len(u1)
        end = self._ptr + n
        if end <= self.capacity:
            self._a1[self._ptr:end] = a1
            self._a2[self._ptr:end] = a2
            self._u1[self._ptr:end] = u1
            self._u2[self._ptr:end] = u2
        else:
            f = self.capacity - self._ptr
            self._a1[self._ptr:] = a1[:f];  self._a1[:n-f] = a1[f:]
            self._a2[self._ptr:] = a2[:f];  self._a2[:n-f] = a2[f:]
            self._u1[self._ptr:] = u1[:f];  self._u1[:n-f] = u1[f:]
            self._u2[self._ptr:] = u2[:f];  self._u2[:n-f] = u2[f:]
        self._ptr  = end % self.capacity
        self._size = min(self._size + n, self.capacity)

    def sample(self, batch_size):
        assert self._size > 0
        n_rec = int(batch_size * self.recent_ratio)
        n_his = batch_size - n_rec
        win   = min(self._size, max(batch_size * 4, 4096))
        if self._ptr >= win:
            ridx = np.arange(self._ptr - win, self._ptr)
        else:
            ridx = np.concatenate([
                np.arange(self.capacity - (win - self._ptr), self.capacity),
                np.arange(0, self._ptr)
            ])
        idx = np.concatenate([
            np.random.choice(ridx, n_rec, replace=True),
            np.random.randint(0, self._size, n_his)
        ])
        np.random.shuffle(idx)
        return {
            "a1": torch.tensor(self._a1[idx], device=self.device),
            "a2": torch.tensor(self._a2[idx], device=self.device),
            "u1": torch.tensor(self._u1[idx], device=self.device),
            "u2": torch.tensor(self._u2[idx], device=self.device),
        }

    def __len__(self):
        return self._size


# —— 验证 ——
print("=== ReplayBuffer 验证 ===")
_buf = ReplayBuffer(capacity=1000)
_a1  = simplex_sample(100)
_a2  = simplex_sample(100)
_u1, _u2 = payoff(_a1, _a2)
_buf.push(_a1, _a2, _u1, _u2)
print(f"Push 100 samples → buffer size: {len(_buf)}")

_batch = _buf.sample(32)
print(f"Sample 32 → a1 shape: {_batch['a1'].shape}  (期望 [32, 3])")
print(f"            u1 shape: {_batch['u1'].shape}  (期望 [32])")
print(f"            u1 范围:  [{_batch['u1'].min():.2f}, {_batch['u1'].max():.2f}]  (应在 [1, 2])")
print(f"            a1 和:    min={_batch['a1'].sum(-1).min():.4f}  max={_batch['a1'].sum(-1).max():.4f}  (应全为1)")

## 4. Hyperparameters & Convergence Criterion

Convergence requires **both** conditions simultaneously:
- `Exploitability < 0.05` (no player can gain > 5% by deviating)
- `Q-Std < 0.05` (Q-function near-flat = indifference condition)

In [ ]:
@dataclass
class Config:
    noise_dim:               int   = 8

    # 预热
    warmup_samples:          int   = 300_000
    warmup_batch_size:       int   = 2048
    warmup_epochs:           int   = 20

    # 交替训练
    total_steps:             int   = 10_000
    actor_batch_size:        int   = 1024
    critic_updates_per_step: int   = 20
    critic_batch_size:       int   = 1024
    actor_update_interval:   int   = 2

    # Q 函数估计
    K_monte_carlo:           int   = 32

    # 损失权重
    beta_ucb:                float = 0.3

    # 优化器
    actor_lr:                float = 1e-4
    critic_lr:               float = 3e-4

    # 经验池
    buffer_capacity:         int   = 200_000

    # 日志与收敛
    log_every:               int   = 200
    converge_exp:            float = 0.05
    converge_qstd:           float = 0.05
    check_cooldown:          int   = 200

    device:                  str   = DEVICE

cfg = Config()
print(cfg)

In [ ]:
def strict_convergence_check(trainer, n=20_000) -> bool:
    cfg = trainer.cfg
    with torch.no_grad():
        a1 = trainer.actor.sample(n, cfg.device)
        a2 = trainer.actor.sample(n, cfg.device)

    gap = nash_gap(a1, a2)
    exp = trainer._exploitability(n_actor=n, n_grid=5000)

    print(f"  Nash Gap:       {gap:.4f}  (< 0.1)")
    print(f"  Exploitability: {exp:.4f}  (< 0.05)")

    converged = (gap < 0.1) and (exp < 0.05)
    if converged:
        trainer.best_actor_state  = copy.deepcopy(trainer.actor.state_dict())
        trainer.best_critic_state = copy.deepcopy(trainer.dc.state_dict())
        trainer.best_step         = len(trainer.history["step"])
        trainer.best_gap          = gap
        trainer.best_exp          = exp
        print(f"  ✅ Converged. Best weights saved at step {trainer.best_step}.")
    else:
        print(f"  ❌ Not converged, continue training.")
    return converged

## 5. Trainer (Single Shared Actor)

Both players share one Actor network (symmetric game assumption).  
Training alternates between:
- **Critic phase** (high-frequency): fit Q(a; π) using K-sample MC targets
- **Actor phase** (low-frequency): maximise Q_cons via replicator dynamics  
  Loss = −(Q_cons − Q̄)  [zero-mean prevents trivial constant-Q collapse]

**Critic warmup:** pre-trains Q on random uniform actions before actor training begins.

In [ ]:
class Trainer:
    def __init__(self, cfg: Config):
        self.cfg        = cfg
        dev             = cfg.device
        self.actor      = Actor(cfg.noise_dim).to(dev)
        self.dc         = DoubleQCritic().to(dev)
        self.opt_actor  = Adam(self.actor.parameters(),  lr=cfg.actor_lr)
        self.opt_critic = Adam(self.dc.parameters(),     lr=cfg.critic_lr)
        self.buffer     = ReplayBuffer(cfg.buffer_capacity, device=dev)
        self.history    = {k: [] for k in
                           ["step","critic_loss","actor_loss",
                            "nash_gap","exploitability","q_std","critic_mae"]}
        self._last_check_step = 0

    # ── 阶段 0：Critic 预热 ───────────────────────────────────────────
    def warmup_critic(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 0: Critic Warmup")

        n_batches = cfg.warmup_samples // cfg.warmup_batch_size
        for _ in range(n_batches):
            a1 = simplex_sample(cfg.warmup_batch_size, cfg.device)
            a2 = simplex_sample(cfg.warmup_batch_size, cfg.device)
            u1, u2 = payoff(a1, a2)
            self.buffer.push(a1, a2, u1, u2)
        print(f"  Buffer size: {len(self.buffer):,}")

        total    = cfg.warmup_epochs * (cfg.warmup_samples // cfg.critic_batch_size)
        log_step = max(total // 10, 1)
        for it in range(total):
            loss = self._update_critic()
            if (it + 1) % log_step == 0:
                print(f"  Warmup [{it+1:5d}/{total}]  Critic Loss: {loss:.6f}")
        print("  Warmup complete.\n")

    # ── 阶段 1：Actor-Critic 交替训练 ─────────────────────────────────
    def train(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 1: Actor-Critic (Replicator Dynamics)")
        print("Convergence: Exploitability -> 0,  Q-Std -> 0")
        print("=" * 60)

        for step in range(1, cfg.total_steps + 1):
            with torch.no_grad():
                a1 = self.actor.sample(cfg.actor_batch_size, cfg.device)
                a2 = self.actor.sample(cfg.actor_batch_size, cfg.device)
            u1, u2 = payoff(a1, a2)
            self.buffer.push(a1, a2, u1, u2)

            c_losses = [self._update_critic()
                        for _ in range(cfg.critic_updates_per_step)]
            c_loss   = sum(c_losses) / len(c_losses)

            if step % cfg.actor_update_interval == 0:
                a_loss, q_std = self._update_actor()
            else:
                a_loss = self.history["actor_loss"][-1] if self.history["actor_loss"] else 0.
                q_std  = self.history["q_std"][-1]      if self.history["q_std"]      else 0.

            with torch.no_grad():
                ae1 = self.actor.sample(4096, cfg.device)
                ae2 = self.actor.sample(4096, cfg.device)
            gap = nash_gap(ae1, ae2)
            exp = self._exploitability()
            mae = self._critic_mae()

            for k, v in zip(
                ["step","critic_loss","actor_loss","nash_gap",
                 "exploitability","q_std","critic_mae"],
                [step, c_loss, a_loss, gap, exp, q_std, mae]
            ):
                self.history[k].append(v)

            if step % cfg.log_every == 0:
                print(f"Step {step:5d} | "
                      f"C-Loss {c_loss:.5f} | "
                      f"C-MAE {mae:.4f} | "
                      f"Exp {exp:.4f} | "
                      f"Q-Std {q_std:.4f} | "
                      f"NashGap {gap:.4f}")

            if (exp   < cfg.converge_exp  and
                q_std < cfg.converge_qstd and
                step  > 500               and
                step - self._last_check_step >= cfg.check_cooldown):
                self._last_check_step = step
                print(f"\nStep={step}, Exp={exp:.4f}, Q-Std={q_std:.4f} -- running strict check...")
                if strict_convergence_check(self):
                    break

        if hasattr(self, "best_actor_state"):
            self.actor.load_state_dict(self.best_actor_state)
            self.dc.load_state_dict(self.best_critic_state)
            print(f"\nRestored best weights from step {self.best_step} "
                  f"(NashGap={self.best_gap:.4f}, Exp={self.best_exp:.4f})")
        print("\nTraining complete.")

    # ── Critic 更新 ───────────────────────────────────────────────────
    def _update_critic(self):
        cfg = self.cfg
        K   = cfg.K_monte_carlo

        batch = self.buffer.sample(cfg.critic_batch_size)
        a1    = batch["a1"]   # (N, 3)

        with torch.no_grad():
            a1_rep    = a1.unsqueeze(1).expand(-1, K, -1).reshape(-1, N_BATTLEFIELDS)
            a2_rep    = self.actor.sample(a1_rep.shape[0], cfg.device)
            u1_rep, _ = payoff(a1_rep, a2_rep)
            q_target  = u1_rep.reshape(-1, K).mean(dim=1)   # (N,)

        q1, q2 = self.dc(a1)
        loss   = F.huber_loss(q1, q_target) + F.huber_loss(q2, q_target)

        self.opt_critic.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.dc.parameters(), 5.)
        self.opt_critic.step()
        return loss.item()

    # ── Actor 更新（复制者动态）──────────────────────────────────────
    def _update_actor(self):
        cfg = self.cfg
        z   = torch.randn(cfg.actor_batch_size, self.actor.noise_dim,
                          device=cfg.device)
        a1  = self.actor(z)

        for p in self.dc.parameters():
            p.requires_grad_(False)

        q_cons    = self.dc.conservative_Q(a1, beta=cfg.beta_ucb)
        q_bar     = q_cons.detach().mean()
        advantage = q_cons - q_bar
        loss      = -advantage.mean()

        self.opt_actor.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 2.)
        self.opt_actor.step()

        for p in self.dc.parameters():
            p.requires_grad_(True)

        q_std = q_cons.detach().std().item()
        return loss.item(), q_std

    # ── 可利用度 ──────────────────────────────────────────────────────
    def _exploitability(self, n_actor=4096, n_grid=2000) -> float:
        with torch.no_grad():
            a1     = self.actor.sample(n_actor, self.cfg.device)
            q_mean = self.dc.mean_Q(a1).mean().item()
            a_grid = simplex_sample(n_grid, self.cfg.device)
            q_max  = self.dc.mean_Q(a_grid).max().item()
        return max(0., q_max - q_mean)

    # ── Critic MAE ────────────────────────────────────────────────────
    def _critic_mae(self, n=1000, K=64) -> float:
        cfg = self.cfg
        with torch.no_grad():
            a1        = simplex_sample(n, cfg.device)
            a1_rep    = a1.unsqueeze(1).expand(-1, K, -1).reshape(-1, N_BATTLEFIELDS)
            a2_rep    = self.actor.sample(a1_rep.shape[0], cfg.device)
            u1_rep, _ = payoff(a1_rep, a2_rep)
            q_true    = u1_rep.reshape(n, K).mean(dim=1)
            q_pred    = self.dc.mean_Q(a1)
        return (q_pred - q_true).abs().mean().item()


# —— 验证 ——
print("=== Trainer 验证 ===")
_cfg     = Config()
_cfg.warmup_samples = 10_000   # 快速验证用
_cfg.warmup_epochs  = 2
_cfg.total_steps    = 5
_trainer = Trainer(_cfg)
_trainer.warmup_critic()
print("Warmup OK")
_trainer.train()
print("Train loop OK")

## 6. Training Run & Visualisation

In [ ]:
cfg     = Config()
trainer = Trainer(cfg)
trainer.warmup_critic()
trainer.train()

In [ ]:
# 可视化
def plot_results(trainer: Trainer):
    h   = trainer.history
    cfg = trainer.cfg
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Colonel Blotto (2P, 3BF) — Actor-Critic Results",
                 fontsize=13, fontweight="bold")

    # 图1：可利用度
    ax = axes[0, 0]
    ax.plot(h["step"], h["exploitability"], color="#e74c3c", lw=1.5)
    ax.axhline(cfg.converge_exp, color="gray", ls="--", label=f"threshold {cfg.converge_exp}")
    ax.set_yscale("log")
    ax.set_xlabel("Step"); ax.set_ylabel("Exploitability")
    ax.set_title("Exploitability"); ax.legend(); ax.grid(alpha=.3)

    # 图2：Q 标准差
    ax = axes[0, 1]
    ax.plot(h["step"], h["q_std"], color="#3498db", lw=1.5)
    ax.axhline(cfg.converge_qstd, color="gray", ls="--", label=f"threshold {cfg.converge_qstd}")
    ax.set_yscale("log")
    ax.set_xlabel("Step"); ax.set_ylabel("Q Std")
    ax.set_title("Q Std"); ax.legend(); ax.grid(alpha=.3)

    # 图3：Critic MAE
    ax = axes[0, 2]
    ax.plot(h["step"], h["critic_mae"], color="#2ecc71", lw=1.5)
    ax.set_xlabel("Step"); ax.set_ylabel("Critic MAE")
    ax.set_title("Critic MAE"); ax.grid(alpha=.3)

    # 图4：a1 vs a2 散点图（战场1 vs 战场2）
    ax = axes[1, 0]
    with torch.no_grad():
        samples = trainer.actor.sample(5000, cfg.device).cpu().numpy()
    ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3, color="#9b59b6")
    ax.set_xlabel("Battlefield 1"); ax.set_ylabel("Battlefield 2")
    ax.set_title("Learned strategy\n(a1 vs a2 projection)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=.3)

    # 图5：每个战场分配的边缘分布
    ax = axes[1, 1]
    colors = ["#e74c3c", "#3498db", "#2ecc71"]
    for j in range(3):
        ax.hist(samples[:, j], bins=50, density=True,
                alpha=0.5, color=colors[j], label=f"Battlefield {j+1}")
    ax.axvline(1/3, color="black", ls="--", lw=2, label="Uniform 1/3")
    ax.set_xlabel("Allocation"); ax.set_ylabel("Density")
    ax.set_title("Per-battlefield marginal distribution")
    ax.legend(fontsize=8); ax.grid(alpha=.3)

    # 图6：Nash Gap
    ax = axes[1, 2]
    ax.plot(h["step"], h["nash_gap"], color="#f39c12", lw=1.5)
    ax.axhline(0.1, color="gray", ls="--", label="threshold 0.1")
    ax.set_xlabel("Step"); ax.set_ylabel("Nash Gap")
    ax.set_title("Nash Gap"); ax.legend(); ax.grid(alpha=.3)

    plt.tight_layout()
    plt.show()

plot_results(trainer)

In [ ]:
plot_results(trainer)

## 7. Final Diagnostics

Verify convergence against the analytical solution (Gross & Wagner, 1950).

In [ ]:
def final_diagnostics(trainer: Trainer):
    cfg = trainer.cfg
    print("=" * 60)
    print("Final Diagnostics — Colonel Blotto")
    print("=" * 60)

    with torch.no_grad():
        a1 = trainer.actor.sample(100_000, cfg.device)
        a2 = trainer.actor.sample(100_000, cfg.device)
    u1, _ = payoff(a1, a2)

    gap = abs(u1.mean().item() - EQ_PAYOFF)
    exp = trainer._exploitability(n_actor=10_000, n_grid=5000)

    a_grid = simplex_sample(2000, cfg.device)
    with torch.no_grad():
        q_vals = trainer.dc.mean_Q(a_grid)
    q_std  = q_vals.std().item()
    q_mean = q_vals.mean().item()

    print(f"Equilibrium payoff (analytical): {EQ_PAYOFF:.4f}")
    print(f"Measured payoff:                 {u1.mean().item():.4f}")
    print(f"Nash Gap:                        {gap:.4f}")
    print(f"Exploitability:                  {exp:.4f}")
    print(f"Q mean (should be ~1.5):         {q_mean:.4f}")
    print(f"Q std  (-> 0 = indifference):    {q_std:.4f}")

    # 对称性检验
    samples = a1.cpu().numpy()
    print(f"\n对称性检验（三个战场边缘分布均值，应均为 1/3）:")
    for j in range(3):
        print(f"  Battlefield {j+1}: mean={samples[:,j].mean():.4f}  std={samples[:,j].std():.4f}")
    print("=" * 60)

final_diagnostics(trainer)

In [ ]:
final_diagnostics(trainer)

## 8. Extension: Two Independent Actors

Each player gets its own Actor and Critic — tests whether **symmetry emerges naturally**
without being imposed as a constraint.  
Expected: both actors converge to similar distributions (low KS divergence).

In [ ]:
# 修改为两个策略网络（两个玩家采用不同网络）测试算法稳定性
class Trainer2Actor:
    """
    两个独立 Actor + 两个独立 Critic。
    玩家1的 Critic 估计 Q(a1; π2)，
    玩家2的 Critic 估计 Q(a2; π1)。
    其余逻辑与单网络版本完全一致。
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        dev = cfg.device

        # 两个独立 Actor，不同随机初始化
        self.actor1 = Actor(cfg.noise_dim).to(dev)
        self.actor2 = Actor(cfg.noise_dim).to(dev)
        self.opt_a1 = Adam(self.actor1.parameters(), lr=cfg.actor_lr)
        self.opt_a2 = Adam(self.actor2.parameters(), lr=cfg.actor_lr)

        # 两个独立 Critic
        self.dc1 = DoubleQCritic().to(dev)
        self.dc2 = DoubleQCritic().to(dev)
        self.opt_c1 = Adam(self.dc1.parameters(), lr=cfg.critic_lr)
        self.opt_c2 = Adam(self.dc2.parameters(), lr=cfg.critic_lr)

        self.buffer = ReplayBuffer(cfg.buffer_capacity, device=dev)
        self.history = {k: [] for k in
                        ["step", "critic_loss", "actor_loss",
                         "nash_gap", "exploitability", "q_std", "critic_mae"]}
        self._last_check_step = 0

    # ── 预热：两个 Critic 同时预热 ────────────────────────────────
    def warmup_critic(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 0: Critic Warmup (2-Actor version)")

        n_batches = cfg.warmup_samples // cfg.warmup_batch_size
        for _ in range(n_batches):
            a1 = simplex_sample(cfg.warmup_batch_size, cfg.device)
            a2 = simplex_sample(cfg.warmup_batch_size, cfg.device)
            u1, u2 = payoff(a1, a2)
            self.buffer.push(a1, a2, u1, u2)
        print(f"  Buffer size: {len(self.buffer):,}")

        total    = cfg.warmup_epochs * (cfg.warmup_samples // cfg.critic_batch_size)
        log_step = max(total // 10, 1)
        for it in range(total):
            l1 = self._update_critic(player=1)
            l2 = self._update_critic(player=2)
            if (it + 1) % log_step == 0:
                print(f"  Warmup [{it+1:5d}/{total}]  "
                      f"C1 Loss: {l1:.6f}  C2 Loss: {l2:.6f}")
        print("  Warmup complete.\n")

    # ── 主训练循环 ────────────────────────────────────────────────
    def train(self):
        cfg = self.cfg
        print("=" * 60)
        print("Phase 1: Actor-Critic, 2 independent actors")
        print("=" * 60)

        for step in range(1, cfg.total_steps + 1):

            # 采集：各自用自己的 Actor
            with torch.no_grad():
                a1 = self.actor1.sample(cfg.actor_batch_size, cfg.device)
                a2 = self.actor2.sample(cfg.actor_batch_size, cfg.device)
            u1, u2 = payoff(a1, a2)
            self.buffer.push(a1, a2, u1, u2)

            # 两个 Critic 各自高频更新
            c_losses = []
            for _ in range(cfg.critic_updates_per_step):
                l1 = self._update_critic(player=1)
                l2 = self._update_critic(player=2)
                c_losses.append((l1 + l2) / 2)
            c_loss = sum(c_losses) / len(c_losses)

            # 两个 Actor 各自更新
            if step % cfg.actor_update_interval == 0:
                a_loss1, q_std1 = self._update_actor(player=1)
                a_loss2, q_std2 = self._update_actor(player=2)
                a_loss = (a_loss1 + a_loss2) / 2
                q_std  = (q_std1  + q_std2)  / 2
            else:
                a_loss = self.history["actor_loss"][-1] if self.history["actor_loss"] else 0.
                q_std  = self.history["q_std"][-1]      if self.history["q_std"]      else 0.

            # 监测指标
            with torch.no_grad():
                ae1 = self.actor1.sample(4096, cfg.device)
                ae2 = self.actor2.sample(4096, cfg.device)
            gap = nash_gap(ae1, ae2)
            exp = self._exploitability()
            mae = self._critic_mae()

            for k, v in zip(
                ["step","critic_loss","actor_loss","nash_gap",
                 "exploitability","q_std","critic_mae"],
                [step, c_loss, a_loss, gap, exp, q_std, mae]
            ):
                self.history[k].append(v)

            if step % cfg.log_every == 0:
                print(f"Step {step:5d} | "
                      f"C-Loss {c_loss:.5f} | "
                      f"C-MAE {mae:.4f} | "
                      f"Exp {exp:.4f} | "
                      f"Q-Std {q_std:.4f} | "
                      f"NashGap {gap:.4f}")

            if (exp   < cfg.converge_exp  and
                q_std < cfg.converge_qstd and
                step  > 500               and
                step - self._last_check_step >= cfg.check_cooldown):
                self._last_check_step = step
                print(f"\nStep={step}, Exp={exp:.4f}, Q-Std={q_std:.4f} -- strict check...")
                if self._strict_check():
                    break

        if hasattr(self, "best_actor1_state"):
            self.actor1.load_state_dict(self.best_actor1_state)
            self.actor2.load_state_dict(self.best_actor2_state)
            self.dc1.load_state_dict(self.best_dc1_state)
            self.dc2.load_state_dict(self.best_dc2_state)
            print(f"\nRestored best weights from step {self.best_step} "
                  f"(NashGap={self.best_gap:.4f}, Exp={self.best_exp:.4f})")
        print("\nTraining complete.")

    # ── Critic 更新（player=1 或 2）───────────────────────────────
    def _update_critic(self, player: int):
        cfg = self.cfg
        K   = cfg.K_monte_carlo

        batch = self.buffer.sample(cfg.critic_batch_size)

        if player == 1:
            a_self  = batch["a1"]       # 玩家1的动作
            dc      = self.dc1
            opt     = self.opt_c1
            # 用当前 actor2 生成对手动作
            opp_actor = self.actor2
        else:
            a_self  = batch["a2"]
            dc      = self.dc2
            opt     = self.opt_c2
            opp_actor = self.actor1

        with torch.no_grad():
            a_rep     = a_self.unsqueeze(1).expand(-1, K, -1).reshape(-1, N_BATTLEFIELDS)
            a_opp     = opp_actor.sample(a_rep.shape[0], cfg.device)
            if player == 1:
                u_rep, _ = payoff(a_rep, a_opp)
            else:
                _, u_rep = payoff(a_opp, a_rep)
            q_target = u_rep.reshape(-1, K).mean(dim=1)

        q1, q2 = dc(a_self)
        loss   = F.huber_loss(q1, q_target) + F.huber_loss(q2, q_target)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dc.parameters(), 5.)
        opt.step()
        return loss.item()

    # ── Actor 更新（player=1 或 2）───────────────────────────────
    def _update_actor(self, player: int):
        cfg = self.cfg
        if player == 1:
            actor = self.actor1
            dc    = self.dc1
            opt   = self.opt_a1
        else:
            actor = self.actor2
            dc    = self.dc2
            opt   = self.opt_a2

        z  = torch.randn(cfg.actor_batch_size, actor.noise_dim, device=cfg.device)
        a  = actor(z)

        for p in dc.parameters():
            p.requires_grad_(False)

        q_cons    = dc.conservative_Q(a, beta=cfg.beta_ucb)
        q_bar     = q_cons.detach().mean()
        loss      = -(q_cons - q_bar).mean()

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(), 2.)
        opt.step()

        for p in dc.parameters():
            p.requires_grad_(True)

        return loss.item(), q_cons.detach().std().item()

    # ── 可利用度（两个玩家取平均）────────────────────────────────
    def _exploitability(self, n_actor=4096, n_grid=2000) -> float:
        with torch.no_grad():
            a1      = self.actor1.sample(n_actor, self.cfg.device)
            a2      = self.actor2.sample(n_actor, self.cfg.device)
            q1_mean = self.dc1.mean_Q(a1).mean().item()
            q2_mean = self.dc2.mean_Q(a2).mean().item()
            a_grid  = simplex_sample(n_grid, self.cfg.device)
            q1_max  = self.dc1.mean_Q(a_grid).max().item()
            q2_max  = self.dc2.mean_Q(a_grid).max().item()
        exp1 = max(0., q1_max - q1_mean)
        exp2 = max(0., q2_max - q2_mean)
        return (exp1 + exp2) / 2

    # ── Critic MAE ────────────────────────────────────────────────
    def _critic_mae(self, n=1000, K=64) -> float:
        cfg = self.cfg
        with torch.no_grad():
            a1        = simplex_sample(n, cfg.device)
            a1_rep    = a1.unsqueeze(1).expand(-1, K, -1).reshape(-1, N_BATTLEFIELDS)
            a2_rep    = self.actor2.sample(a1_rep.shape[0], cfg.device)
            u1_rep, _ = payoff(a1_rep, a2_rep)
            q_true    = u1_rep.reshape(n, K).mean(dim=1)
            q_pred    = self.dc1.mean_Q(a1)
        return (q_pred - q_true).abs().mean().item()

    # ── 严格收敛检验 ──────────────────────────────────────────────
    def _strict_check(self, n=20_000) -> bool:
        cfg = self.cfg
        with torch.no_grad():
            a1 = self.actor1.sample(n, cfg.device)
            a2 = self.actor2.sample(n, cfg.device)
        gap = nash_gap(a1, a2)
        exp = self._exploitability(n_actor=n, n_grid=5000)

        print(f"  Nash Gap:       {gap:.4f}  (< 0.1)")
        print(f"  Exploitability: {exp:.4f}  (< 0.05)")

        converged = (gap < 0.1) and (exp < 0.05)
        if converged:
            self.best_actor1_state = copy.deepcopy(self.actor1.state_dict())
            self.best_actor2_state = copy.deepcopy(self.actor2.state_dict())
            self.best_dc1_state    = copy.deepcopy(self.dc1.state_dict())
            self.best_dc2_state    = copy.deepcopy(self.dc2.state_dict())
            self.best_step         = len(self.history["step"])
            self.best_gap          = gap
            self.best_exp          = exp
            print(f"  ✅ Converged.")
        else:
            print(f"  ❌ Not converged.")
        return converged

In [ ]:
cfg2     = Config()
trainer2 = Trainer2Actor(cfg2)
trainer2.warmup_critic()
trainer2.train()

In [ ]:
def plot_comparison(t1: Trainer, t2: Trainer2Actor):
    """
    左列：单网络版本
    右列：双网络版本
    对比策略分布和训练曲线
    """
    fig, axes = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle("Single Actor vs Two Independent Actors", fontsize=13, fontweight="bold")

    with torch.no_grad():
        s1      = t1.actor.sample(10_000, t1.cfg.device).cpu().numpy()
        s2_p1   = t2.actor1.sample(10_000, t2.cfg.device).cpu().numpy()
        s2_p2   = t2.actor2.sample(10_000, t2.cfg.device).cpu().numpy()

    colors = ["#e74c3c", "#3498db", "#2ecc71"]

    # 行1：单网络边缘分布 | 双网络边缘分布（两玩家叠加）
    ax = axes[0, 0]
    for j in range(3):
        ax.hist(s1[:, j], bins=50, density=True, alpha=0.5,
                color=colors[j], label=f"BF {j+1}")
    ax.axvline(1/3, color="black", ls="--", lw=2, label="1/3")
    ax.set_title("Single Actor — marginal distribution")
    ax.set_xlabel("Allocation"); ax.legend(fontsize=8); ax.grid(alpha=.3)

    ax = axes[0, 1]
    for j in range(3):
        ax.hist(s2_p1[:, j], bins=50, density=True, alpha=0.4,
                color=colors[j], label=f"P1 BF{j+1}")
        ax.hist(s2_p2[:, j], bins=50, density=True, alpha=0.2,
                color=colors[j], linestyle="--")
    ax.axvline(1/3, color="black", ls="--", lw=2, label="1/3")
    ax.set_title("Two Actors — marginal distribution\n(solid=P1, faded=P2)")
    ax.set_xlabel("Allocation"); ax.legend(fontsize=7); ax.grid(alpha=.3)

    # 行2：a1-a2 散点投影
    ax = axes[1, 0]
    ax.scatter(s1[:, 0], s1[:, 1], s=2, alpha=0.3, color="#9b59b6")
    ax.set_title("Single Actor — a1 vs a2 projection")
    ax.set_xlabel("BF1"); ax.set_ylabel("BF2")
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=.3)

    ax = axes[1, 1]
    ax.scatter(s2_p1[:, 0], s2_p1[:, 1], s=2, alpha=0.3,
               color="#e74c3c", label="Player 1")
    ax.scatter(s2_p2[:, 0], s2_p2[:, 1], s=2, alpha=0.3,
               color="#3498db", label="Player 2")
    ax.set_title("Two Actors — a1 vs a2 projection")
    ax.set_xlabel("BF1"); ax.set_ylabel("BF2")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(fontsize=8); ax.grid(alpha=.3)

    # 行3：Exploitability 训练曲线对比
    ax = axes[2, 0]
    ax.plot(t1.history["step"], t1.history["exploitability"],
            color="#e74c3c", lw=1.5, label="Single Actor")
    ax.plot(t2.history["step"], t2.history["exploitability"],
            color="#3498db", lw=1.5, label="Two Actors")
    ax.axhline(cfg.converge_exp, color="gray", ls="--", label="threshold")
    ax.set_yscale("log")
    ax.set_xlabel("Step"); ax.set_ylabel("Exploitability")
    ax.set_title("Exploitability comparison")
    ax.legend(); ax.grid(alpha=.3)

    # 行3右：Nash Gap 对比
    ax = axes[2, 1]
    ax.plot(t1.history["step"], t1.history["nash_gap"],
            color="#e74c3c", lw=1.5, label="Single Actor")
    ax.plot(t2.history["step"], t2.history["nash_gap"],
            color="#3498db", lw=1.5, label="Two Actors")
    ax.axhline(0.1, color="gray", ls="--", label="threshold")
    ax.set_xlabel("Step"); ax.set_ylabel("Nash Gap")
    ax.set_title("Nash Gap comparison")
    ax.legend(); ax.grid(alpha=.3)

    plt.tight_layout()
    plt.show()

plot_comparison(trainer, trainer2)

In [ ]:
plot_comparison(trainer, trainer2)

In [ ]:
def compare_diagnostics(t1: Trainer, t2: Trainer2Actor):
    cfg = t1.cfg
    print("=" * 60)
    print("Comparison: Single Actor vs Two Actors")
    print("=" * 60)

    # 单网络
    with torch.no_grad():
        a1 = t1.actor.sample(100_000, cfg.device)
        a2 = t1.actor.sample(100_000, cfg.device)
    u1_s, _ = payoff(a1, a2)
    s = a1.cpu().numpy()

    # 双网络
    with torch.no_grad():
        b1 = t2.actor1.sample(100_000, cfg.device)
        b2 = t2.actor2.sample(100_000, cfg.device)
    u1_d, _ = payoff(b1, b2)
    d1 = b1.cpu().numpy()
    d2 = b2.cpu().numpy()

    print(f"\n{'指标':<30} {'单网络':>12} {'双网络':>12}")
    print("-" * 55)
    print(f"{'Nash Gap':<30} {abs(u1_s.mean().item()-EQ_PAYOFF):>12.4f} "
          f"{abs(u1_d.mean().item()-EQ_PAYOFF):>12.4f}")
    print(f"{'Exploitability':<30} {t1._exploitability():>12.4f} "
          f"{t2._exploitability():>12.4f}")
    print(f"{'收敛步数':<30} {t1.history['step'][-1]:>12d} "
          f"{t2.history['step'][-1]:>12d}")

    print(f"\n对称性检验（边缘均值，理论值 = 0.333）:")
    print(f"{'':10} {'BF1':>8} {'BF2':>8} {'BF3':>8}  {'最大偏差':>10}")
    means_s = s.mean(0)
    means_d1 = d1.mean(0)
    means_d2 = d2.mean(0)
    print(f"{'单网络':<10} {means_s[0]:>8.4f} {means_s[1]:>8.4f} {means_s[2]:>8.4f}"
          f"  {abs(means_s - 1/3).max():>10.4f}")
    print(f"{'双网络P1':<10} {means_d1[0]:>8.4f} {means_d1[1]:>8.4f} {means_d1[2]:>8.4f}"
          f"  {abs(means_d1 - 1/3).max():>10.4f}")
    print(f"{'双网络P2':<10} {means_d2[0]:>8.4f} {means_d2[1]:>8.4f} {means_d2[2]:>8.4f}"
          f"  {abs(means_d2 - 1/3).max():>10.4f}")

    # 两个玩家分布相似度（KS检验）
    print(f"\n双网络两玩家分布相似度（KS统计量，越小越相似）:")
    for j in range(3):
        ks, _ = stats.ks_2samp(d1[:, j], d2[:, j])
        print(f"  Battlefield {j+1}: KS = {ks:.4f}")

compare_diagnostics(trainer, trainer2)

In [ ]:
compare_diagnostics(trainer, trainer2)